In [1]:
import json
import glob
import os
import pandas as pd
import numpy as np

In [2]:
RAW_DIR = "..\\data"

# 1. Load ONLY the raw movie files (ignoring the keyword file)
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.jsonl")))
print(f"Found {len(files)} raw movie files.")

rows = []
for f in files:
    with open(f, "r", encoding="utf-8") as file:
        rows.extend([json.loads(line) for line in file if line.strip()])

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} movies with {df.shape[1]} columns.")

Found 1 raw movie files.
Loaded 10639 movies with 27 columns.


In [4]:
print("budget == 0:", (df["budget"] == 0).sum(), f"({(df['budget'] == 0).mean()*100:.1f}%)")

print("revenue == 0:", (df["revenue"] == 0).sum(), f"({(df['revenue'] == 0).mean()*100:.1f}%)")

print("both == 0:", ((df["budget"] == 0) & (df["revenue"] == 0)).sum())

budget == 0: 3075 (28.9%)
revenue == 0: 1483 (13.9%)
both == 0: 1260


In [6]:
df[df["budget"] == 0][["title", "release_date", "revenue", "vote_count"]].head(10)

,title,release_date,revenue,vote_count
18,Seems Like Old Times,1980-12-19,43995918,172
20,Cheech & Chong's Next Movie,1980-07-18,41675194,288
27,Doraemon: Nobita's Dinosaur,1980-03-15,25000000,74
31,My Bodyguard,1980-07-11,22482952,140
37,Herbie Goes Bananas,1980-06-25,18000000,264
46,The Mirror Crack'd,1980-02-06,11000000,271
50,Tom Horn,1980-03-28,9000000,139
51,The Awakening,1980-07-09,8415112,96
55,La Cage aux Folles II,1980-12-10,6950125,122
56,Where the Buffalo Roam,1980-04-25,6659377,194


In [ ]:
budget_zeros_before = (df["budget"] == 0).sum()
budget_nulls_before = (df["budget"].isnull()).sum()
revenue_zeros_before = (df["revenue"] == 0).sum()
revenue_nulls_before = (df["revenue"].isnull()).sum()

df["budget"] = df["budget"].replace(0, np.nan)
df["revenue"] = df["revenue"].replace(0, np.nan)

print("budget nulls after fix:",(df["budget"].isna()).sum(), f"({(df['budget'].isna()).mean() * 100:.1f}%)")
print("revenue nulls after fix:", (df["revenue"].isna()).sum(), f"({(df['revenue'].isna()).mean() * 100:.1f}%)")

budget nulls after fix: 3075 (28.9%)
revenue nulls after fix: 1483 (13.9%)
Sanity checks passed.


In [8]:
both_missing = df["budget"].isna() & df["revenue"].isna()
print("both missing:", both_missing.sum())

budget_only_missing = df["budget"].isna() & df["revenue"].notna()
revenue_only_missing = df["budget"].notna() & df["revenue"].isna()
print(f"budget missing, revenue known: {budget_only_missing.sum()}")
print(f"budget known, revenue missing: {revenue_only_missing.sum()}")

both missing: 1260
budget missing, revenue known: 1815
budget known, revenue missing: 223


TMDB uses 0 for missing budgets, not just free films. I replace 0 with NaN and verify the null counts.

In [ ]:
df["budget"] = df["budget"].replace(0, np.nan)
df["revenue"] = df["revenue"].replace(0, np.nan)

n_zero = (df["vote_count"] == 0).sum()
df = df[df["vote_count"] > 0].copy()
print(f"Removed {n_zero} rows with 0 vote_count. New shape: {df.shape}")

df = df[df["vote_count"] >= 100].copy()
print(f'After vote filter (>=100), new shape: {df.shape}. Length of df: {len(df)}')

Removed 0 rows with 0 vote_count. New shape: (10639, 27)
After vote filter (>=100), new shape: (9517, 27). Length of df: 9517


Before writing any functions, we need to see the actual shape of the nested data

In [10]:
print("genres sample:", df["genres"].iloc[0])
print("genres type:", type(df["genres"].iloc[0]))

print("\nbelongs_to_collection sample:", df["belongs_to_collection"].iloc[0])
print("collection type:", type(df["belongs_to_collection"].iloc[0]))

print("\nrelease_date sample:", repr(df["release_date"].iloc[0]))

# How many rows have null collection?
print("\ncollection nulls:", df["belongs_to_collection"].isna().sum())

genres sample: [{'id': 12, 'name': 'Adventure'}, {'id': 28, 'name': 'Action'}, {'id': 878, 'name': 'Science Fiction'}]
genres type: <class 'list'>

belongs_to_collection sample: {'id': 10, 'name': 'Star Wars Collection', 'poster_path': '/pWVLFh4OuejTpUaDQbB1C4zoS2p.jpg', 'backdrop_path': '/iY2ujEY2m68OTTlPFTiHub9joHS.jpg'}
collection type: <class 'dict'>

release_date sample: '1980-05-20'

collection nulls: 6855


Extract Nested Fields

In [ ]:
def primary_genre(x):
    if isinstance(x, list) and len(x) >0:
        return x[0]["name"]
    return None

df["primary_genre"] = df["genres"].apply(primary_genre)

def collection_id(x):
    if isinstance(x, dict):
        return x.get("id")
    return None

df["collection_id"] = df["belongs_to_collection"].apply(collection_id)

df["is_franchise"] = df["collection_id"].notna()

df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype("Int64")




primary_genre nulls: 0
is_franchise: 2662 (28.0%)
release_year range: 1980 – 2025
release_year nulls: 0


In [12]:
valid_financials = df["budget"].notna() & df["revenue"].notna()

df["roi"] = np.where(valid_financials & (df["budget"] > 0), df["revenue"] / df["budget"], np.nan)
df["profit"] = np.where(valid_financials, df["revenue"] - df["budget"], np.nan)
df["log_roi"] = np.where((df["roi"] > 0), np.log(df["roi"]), np.nan)

print(f"Movies with valid financials: {valid_financials.sum()}")
print(f"Movies with roi: {df['roi'].notna().sum()}")
assert df["roi"].notna().sum() == valid_financials.sum(), "roi count mismatch"
assert df["profit"].notna().sum() == valid_financials.sum(), "profit count mismatch"
print("Sanity checks passed.")

# Distribution summary
print("\nROI describe:")
print(df["roi"].describe())
print("\nlog_roi describe:")
print(df["log_roi"].describe())

Movies with valid financials: 7039
Movies with roi: 7039
Sanity checks passed.

ROI describe:
count    7.039000e+03
mean     3.560774e+02
std      2.928281e+04
min      2.060000e-05
25%      9.257401e-01
50%      2.084044e+00
75%      4.120005e+00
max      2.456760e+06
Name: roi, dtype: float64

log_roi describe:
count    7039.000000
mean        0.631912
std         1.328948
min       -10.790219
25%        -0.077162
50%         0.734310
75%         1.415854
max        14.714354
Name: log_roi, dtype: float64


In [13]:
df["suspect_roi"] = ((df["budget"] < 10_000) | (df["roi"] < 0.01))
print(f"Suspect ROI flagged: {df['suspect_roi'].sum()} movies ({df['suspect_roi'].mean()*100:.2f}%)")


Suspect ROI flagged: 29 movies (0.30%)


In [14]:
clean_df = df[
    df["budget"].notna() &
    df["revenue"].notna() &
    df["release_year"].notna() &
    (~df["suspect_roi"]) &
    (df["vote_count"] >= 100) # Added the vote filter back
].copy()

In [15]:
print(f"Final clean dataset: {len(clean_df)} movies")


Final clean dataset: 7013 movies


In [16]:
# Drop the messy JSON columns so the CSV is readable
cols_to_drop = ["genres", "belongs_to_collection", "production_companies", "production_countries", "spoken_languages"]
clean_df = clean_df.drop(columns=[c for c in cols_to_drop if c in clean_df.columns])

clean_df.to_csv("../data/movies_clean.csv", index=False)
print("Saved clean base dataset to ../data/movies_clean.csv")

Saved clean base dataset to ../data/movies_clean.csv


In [17]:
import json
import pandas as pd

with open("../data/keywords/movie_keywords.json", "r") as f:
    keywords_dict = json.load(f)
    print(f"Loaded keywords for {len(keywords_dict)} movies")

# (Your comprehensive lists are great, I kept them exactly as you had them)
ADAPTATION_KEYWORDS = [    
    "based on young adult novel", "based on cartoon", "based on song, poem or rhyme",
    "based on play or musical", "based on graphic novel", "based on comic",
    "based on light novel", "based on video game", "based on documentary",
    "based on manga", "based on movie", "based on fairy tale",
    "based on children's book", "based on anime", "based on short film",
    "based on short story", "based on webcomic or webtoon", "based on novel or book",
    "based on myths, legends or folklore", "based on game", "based on web series",
    "based on tv series", "based on memoir or autobiography"
]
REMAKE_KEYWORDS = [
    "remake", "live action remake", "reboot", "reimagining", 
    "reimagined", "rebooted", "remade", "remakes", "remaking", 
]

def check_derivative(kw_list):
    # If it's NaN (because the ID wasn't found in the JSON), return False
    if not isinstance(kw_list, list):
        return False, False

    # Extract names safely, handling BOTH dicts and strings
    kw_names = []
    for k in kw_list:
        if isinstance(k, dict) and "name" in k:
            kw_names.append(k["name"].lower())
        elif isinstance(k, str):
            kw_names.append(k.lower())
            
    # Now check if any target keywords exist in the clean list
    is_adaptation = any(kw in kw_names for kw in ADAPTATION_KEYWORDS)
    is_remake = any(kw in kw_names for kw in REMAKE_KEYWORDS)
    
    return is_adaptation, is_remake

# 1. Map the JSON to the dataframe (using strings)
clean_df["kw_list"] = clean_df["id"].astype(str).map(keywords_dict)

# 2. Apply the function
clean_df[["is_adaptation", "is_remake"]] = clean_df["kw_list"].apply(
    lambda x: pd.Series(check_derivative(x))
)

# 3. Create the master flag
clean_df["is_derivative"] = clean_df["is_adaptation"] | clean_df["is_remake"] | clean_df["is_franchise"]

# 4. Print results
print(f"Franchises (Sequels/Prequels): {clean_df['is_franchise'].mean()*100:.1f}%")
print(f"Adaptations:                   {clean_df['is_adaptation'].mean()*100:.1f}%")
print(f"Remakes:                       {clean_df['is_remake'].mean()*100:.1f}%")
print(f"-------------------------------------------")
print(f"TOTAL Derivative Works:        {clean_df['is_derivative'].mean()*100:.1f}%")

Loaded keywords for 7310 movies
Franchises (Sequels/Prequels): 28.2%
Adaptations:                   22.7%
Remakes:                       4.5%
-------------------------------------------
TOTAL Derivative Works:        45.7%


In [18]:
# 1. Drop the list of keywords so the CSV stays flat and clean
clean_df = clean_df.drop(columns=["kw_list"])

# 2. Save the final, fully-enriched dataset
clean_df.to_csv("../data/movies_final.csv", index=False)

print(f"Successfully saved movies_final.csv with {len(clean_df)} rows and {clean_df.shape[1]} columns.")

Successfully saved movies_final.csv with 7013 rows and 33 columns.
